In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Simulazione esatta con le primitive di Qiskit SDK

<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato con i seguenti requisiti.
Consigliamo di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
```
</details>

Le primitive di riferimento nel Qiskit SDK eseguono simulazioni locali dello statevector. Queste simulazioni non supportano
la modellazione del rumore del dispositivo, ma sono utili per prototipare rapidamente algoritmi prima di esplorare tecniche
di simulazione più avanzate ([usando Qiskit Aer](./simulate-stabilizer-circuits)) o di eseguire su dispositivi reali ([primitive di Qiskit Runtime](primitives)).

La primitiva Estimator può calcolare i valori attesi dei circuiti, mentre la primitiva Sampler può campionare
dalle distribuzioni di output dei circuiti.

Le sezioni seguenti mostrano come usare le primitive di riferimento per eseguire il tuo workflow in locale.

## Usare l'Estimator di riferimento
L'implementazione di riferimento di `EstimatorV2` in `qiskit.primitives` che gira su simulatori locali dello statevector
è la classe [`StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator). Può ricevere circuiti, osservabili e parametri come input e restituisce i valori attesi calcolati localmente.

Il codice seguente prepara gli input che verranno usati negli esempi successivi. Il tipo di input atteso per gli
osservabili è [`qiskit.quantum_info.SparsePauliOp`](../api/qiskit/qiskit.quantum_info.SparsePauliOp). Nota che
il circuito nell'esempio è parametrizzato, ma puoi eseguire l'Estimator anche su circuiti non parametrizzati.

> **Note:** Qualsiasi circuito passato a un Estimator **non** deve includere **misurazioni**.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# circuit for which you want to obtain the expected value
circuit = QuantumCircuit(2)
circuit.ry(Parameter("theta"), 0)
circuit.h(0)
circuit.cx(0, 1)
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/5b41a52d-8f15-4ce4-b3f6-effd91946d9c-0.svg" alt="Output of the previous code cell" />

In [2]:
from qiskit.quantum_info import SparsePauliOp
import numpy as np

# observable(s) whose expected values you want to compute

observable = SparsePauliOp(["II", "XX", "YY", "ZZ"], coeffs=[1, 1, -1, 1])

# value(s) for the circuit parameter(s)
parameter_values = [[0], [np.pi / 6], [np.pi / 2]]

> **Tip:** Il workflow delle primitive di Qiskit Runtime richiede che circuiti e osservabili vengano trasformati in modo da usare solo le istruzioni supportate dalla QPU (chiamati circuiti e osservabili con *instruction set architecture (ISA)*). Le primitive di riferimento accettano ancora istruzioni astratte, poiché si basano su simulazioni locali dello statevector, ma eseguire il transpiling del circuito può comunque essere vantaggioso in termini di ottimizzazione.
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(circuit)
>   isa_observable = observable.apply_layout(isa_circuit.layout)
>   ```

### Inizializzare l'Estimator
Istanzia un [`qiskit.primitives.StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator).

In [3]:
from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator()

### Eseguire e ottenere i risultati
Questo esempio usa un solo circuito (di tipo [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit)) e un solo
osservabile.

Esegui la stima chiamando il metodo [`StatevectorEstimator.run`](../api/qiskit/qiskit.primitives.StatevectorEstimator#run), che restituisce un'istanza di un oggetto [`PrimitiveJob`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveJob). Puoi ottenere i risultati dal job (come oggetto [`qiskit.primitives.PrimitiveResult`](../api/qiskit/qiskit.primitives.PrimitiveResult))
con il metodo [`qiskit.primitives.PrimitiveJob.result`](../api/qiskit/qiskit.primitives.PrimitiveJob#result).

In [4]:
job = estimator.run([(circuit, observable, parameter_values)])
result = job.result()
print(f" > Result class: {type(result)}")

 > Result class: <class 'qiskit.primitives.containers.primitive_result.PrimitiveResult'>


#### Get the expected value from the result

The primitives result outputs an array of [`PubResult`](/docs/api/qiskit/qiskit.primitives.PubResult#pubresult) objects, where each item of the array is a `PubResult` object that contains in its data the array of evaluations corresponding to every circuit-observable combination in the PUB.

To retrieve the expectation values and metadata for the first (and in this case, only) circuit evaluation, we must access the evaluation [`data`](/docs/api/qiskit/qiskit.primitives.PubResult#data) for PUB 0:

In [5]:
print(f" > Expectation value: {result[0].data.evs}")
print(f" > Metadata: {result[0].metadata}")

 > Expectation value: [4.         3.73205081 2.        ]
 > Metadata: {'target_precision': 0.0, 'circuit_metadata': {}}


#### Ottenere il valore atteso dal risultato
Il risultato delle primitive restituisce un array di oggetti [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult), dove ogni elemento dell'array è un oggetto `PubResult` che contiene nei suoi dati l'array di valutazioni corrispondente a ogni combinazione circuito-osservabile nel PUB.

Per recuperare i valori attesi e i metadati per la prima (e in questo caso unica) valutazione del circuito, dobbiamo accedere ai [`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#data) della valutazione per il PUB 0:

In [6]:
# Estimate expectation values for two PUBs, both with 0.05 precision.
precise_job = estimator.run(
    [(circuit, observable, parameter_values)], precision=0.05
)

For a full example, see the [Primitives examples](primitives-examples#estimator-examples) page.

## Use the reference Sampler

The reference implementations of `SamplerV2` in `qiskit.primitives` is the [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler) class. It takes circuits and parameters as inputs and returns the results from sampling from the output probability distributions as a quasi-probability distribution of output states.

The following code prepares the inputs used in the examples that follow. Note that
these examples run a single parametrized circuit, but you can also run the Sampler
on non-parametrized circuits.

In [7]:
from qiskit import QuantumCircuit

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg" alt="Output of the previous code cell" />

### Impostare le opzioni di esecuzione dell'Estimator
Per impostazione predefinita, l'Estimator di riferimento esegue un calcolo esatto dello statevector basato sulla
classe [`quantum_info.Statevector`](../api/qiskit/qiskit.quantum_info.Statevector).
Tuttavia, questo può essere modificato per introdurre l'effetto dell'overhead di campionamento (noto anche come "shot noise").

L'Estimator accetta un argomento `precision` che esprime le barre di errore che l'implementazione della primitiva
dovrebbe puntare a ottenere per le stime dei valori attesi. Questo rappresenta l'overhead di campionamento ed è definito esclusivamente nel metodo `.run()`. Ciò ti consente di mettere a punto l'opzione fino al livello del PUB.

In [8]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()

Per un esempio completo, consulta la pagina degli [esempi con le primitive](primitives-examples#estimator-examples).

## Usare il Sampler di riferimento
L'implementazione di riferimento di `SamplerV2` in `qiskit.primitives` è la classe [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler). Riceve circuiti e parametri come input e restituisce i risultati del campionamento dalle distribuzioni di probabilità di output come distribuzione di quasi-probabilità degli stati di output.

Il codice seguente prepara gli input usati negli esempi successivi. Nota che
questi esempi eseguono un singolo circuito parametrizzato, ma puoi eseguire il Sampler
anche su circuiti non parametrizzati.

In [9]:
# execute 1 circuit with Sampler
job = sampler.run([circuit])
pub_result = job.result()[0]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


![Output della cella di codice precedente](../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg)

> **Note:** Qualsiasi circuito quantistico passato a un Sampler **deve** includere misurazioni.

> **Tip:** Il workflow delle primitive di Qiskit Runtime richiede che i circuiti vengano trasformati in modo da usare solo le istruzioni supportate dalla QPU (chiamati circuiti ISA). Le primitive di riferimento accettano ancora istruzioni astratte, poiché si basano su simulazioni locali dello statevector, ma eseguire il transpiling del circuito può comunque essere vantaggioso in termini di ottimizzazione.
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(qc)
>   ```

### Inizializzare `SamplerV2`
Istanzia [`qiskit.primitives.StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler):

In [10]:
from qiskit.transpiler import generate_preset_pass_manager

# create two circuits
circuit1 = circuit.copy()
circuit2 = circuit.copy()

# transpile circuits
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit1 = pm.run(circuit1)
isa_circuit2 = pm.run(circuit2)
# execute 2 circuits using Sampler
job = sampler.run([(isa_circuit1), (isa_circuit2)])
pub_result_1 = job.result()[0]
pub_result_2 = job.result()[1]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


### Eseguire e ottenere i risultati

In [11]:
# Define quantum circuit with 2 qubits
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw()

        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 

In [12]:
# Transpile circuit
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit = pm.run(circuit)
# Run using sampler
result = sampler.run([circuit]).result()
# Access result data for PUB 0
data_pub = result[0].data
# Access bitstring for the classical register "meas"
bitstrings = data_pub.meas.get_bitstrings()
print(f"The number of bitstrings is: {len(bitstrings)}")
# Get counts for the classical register "meas"
counts = data_pub.meas.get_counts()
print(f"The counts are: {counts}")

The number of bitstrings is: 1024
The counts are: {'11': 515, '00': 509}


Le primitive accettano più PUB come input e ogni PUB ottiene il proprio risultato. Puoi quindi eseguire circuiti diversi con varie combinazioni di parametri/osservabili e recuperare i risultati dei PUB:

In [13]:
# Sample two circuits at 128 shots each.
sampler.run([isa_circuit1, isa_circuit2], shots=128)
# Sample two circuits at different amounts of shots. The "None"s are necessary
# as placeholders
# for the lack of parameter values in this example.
sampler.run([(isa_circuit1, None, 123), (isa_circuit2, None, 456)])

For a full example, see the [Primitives examples](./primitives-examples#sampler-examples) page.
## Next steps

<Admonition type="tip" title="Recommendations">
  - For higher-performance simulation that can handle larger circuits, or to incorporate noise models into your simulation, see [Exact and noisy simulation with Qiskit Aer primitives](simulate-with-qiskit-aer).
  - To learn how to use Quantum Composer for simulation, see the [IBM Quantum Composer](/docs/guides/composer) guide.
  - Read the [Qiskit Estimator API](/docs/api/qiskit/1.4/qiskit.primitives.Estimator) reference.
  - Read the [Qiskit Sampler API](/docs/api/qiskit/1.4/qiskit.primitives.Sampler) reference.
  - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
</Admonition>